# MLflow

A self-contained refresher on **MLflow** — the open-source platform for managing the machine-learning lifecycle: experiment tracking, model packaging, a model registry, and reproducible runs.

**Domain:** AI/ML Tooling  ·  **runnable:** yes

## 1. What & Why

**MLflow answers the question "which run produced this model, and how do I reproduce and ship it?"** Training a model is a loop of *try a config → get a number*, run dozens of times. Without tooling, that history lives in scrollback, notebook cells, and filenames like `model_final_v3_REAL.pkl`. MLflow turns each attempt into a recorded **run** — its parameters, metrics, code version, and output artifacts — that you can query, compare, and reload months later.

It is **framework-agnostic and unobtrusive**: a few `mlflow.log_*` calls (or one `mlflow.autolog()`) drop into existing scikit-learn / PyTorch / XGBoost / Keras code, and it runs locally with zero infrastructure — a folder or a SQLite file — then scales to a shared tracking server or Databricks without changing your code.

**Reach for MLflow when** you run many experiments and need to compare them, when "what hyperparameters gave that result?" must be answerable, or when you need a clean handoff from training to deployment (a versioned, signature-checked model artifact). **Look elsewhere when** you only ever train one model once (overkill), or you need full data+pipeline versioning and orchestration — that's [`DVC`](dvc.ipynb), Kubeflow, or a workflow engine, not MLflow's core job.

## 2. Mental Model

Think of MLflow as a **lab notebook + a parts warehouse for ML**:

- A **run** is one page of the lab notebook: *here's the recipe I tried (params), the readings I got (metrics), and the stuff it produced (artifacts).*
- An **experiment** is the binder those pages live in — all the runs for one project/question.
- The **tracking store** is the filing cabinet (a local folder, a SQLite/Postgres DB, or a remote server).
- The **Model Registry** is the warehouse where a *chosen* model graduates from "a file in some run" to a **named, versioned product** with stages/aliases (`@champion`, `@challenger`) that deployment systems pull from.

```
   your training code
        │  mlflow.log_param / log_metric / log_model   (or mlflow.autolog())
        ▼
   ┌─────────── RUN ───────────┐        search_runs / MLflow UI
   │ params · metrics · tags   │ ─────────────────────────────▶  compare & pick best
   │ artifacts (model, plots)  │
   └──────────────┬────────────┘
                  │ register_model("name")
                  ▼
        MODEL REGISTRY:  name "iris-classifier"
            v1 · v2 · v3 ...  with aliases  @champion / @staging
                  │  models:/iris-classifier@champion
                  ▼
            load for serving / batch inference
```

The key mental shift: **you don't save files, you log runs.** MLflow owns the bookkeeping so the artifact always travels with the context that explains it.

## 3. Key Concepts

MLflow is four loosely-coupled components — you can use just **Tracking** and ignore the rest:

- **Tracking** — the API + store that records **runs**. A run holds **params** (inputs you set, immutable), **metrics** (numbers that can change over `step`s, e.g. loss per epoch), **tags** (free-form metadata), and **artifacts** (any output file: the model, plots, data samples).
- **Experiment** — a named bucket of runs. Set it with `mlflow.set_experiment("name")`.
- **Models** — MLflow's standard **packaging format**. `log_model` saves a model in *flavors* (e.g. `sklearn` and the universal `python_function`/`pyfunc`) plus an `MLmodel` metadata file, its dependencies, and an optional **signature** (input/output schema). `pyfunc` lets *any* logged model be loaded and called the same way: `model.predict(df)`.
- **Model Registry** — a versioned catalog on top of logged models. `register_model` creates `name/version`; **aliases** (`@champion`) and **tags** point deployment at a specific version. (Requires a database-backed store, e.g. SQLite/Postgres — the bare-folder store can't do registry.)
- **Tracking URI** — *where* runs go: `./mlruns` (default folder), `sqlite:///mlflow.db`, or `http://my-server:5000` / Databricks. Set via `mlflow.set_tracking_uri(...)` or the `MLFLOW_TRACKING_URI` env var. **Same code, different backend.**
- **`autolog()`** — one call (`mlflow.autolog()` or `mlflow.sklearn.autolog()`) that auto-captures params, metrics, and the fitted model for supported libraries — no manual `log_*` calls.
- **Projects** — an `MLproject` file pinning entry points + a Conda/virtualenv environment for reproducible `mlflow run .` execution (less central in day-to-day use than Tracking/Models).

## 4. Setup

MLflow is a pure-Python package; the core needs no GPU and no server.

```bash
pip install mlflow            # full platform (tracking, models, registry, UI, serving)
# pip install mlflow-skinny   # lightweight client-only install (no UI/serving deps)
```

Two ways to view results:
- **Launch the UI:** `mlflow ui` (reads `./mlruns`) or `mlflow ui --backend-store-uri sqlite:///mlflow.db`, then open http://localhost:5000.
- **Query in code:** `mlflow.search_runs(...)` returns a pandas DataFrame — what we use below so the notebook is self-contained.

The cell below points the tracking store at a temp **SQLite** file (no server, no network) so every component — including the Model Registry — works inside this notebook.

In [1]:
import os
import tempfile
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")  # quiet MLflow/sklearn version-hint noise for the refresher

import mlflow

# Local, file-based backend: no server, no network. SQLite (vs. a bare ./mlruns
# folder) is what unlocks the Model Registry used in Example 2.
workdir = Path(tempfile.mkdtemp(prefix="mlflow_demo_"))
mlflow.set_tracking_uri(f"sqlite:///{workdir / 'mlruns.db'}")
mlflow.set_registry_uri(mlflow.get_tracking_uri())

print("MLflow version:", mlflow.__version__)
print("Tracking URI :", mlflow.get_tracking_uri())

MLflow version: 3.14.0
Tracking URI : sqlite:////var/folders/p8/sm5jmh055md_zzhhn1mfgyw80000gn/T/mlflow_demo_m5rrsf5r/mlruns.db


## 5. Worked Examples

### Example 1 — Manual tracking: log a run, then query it back

The core loop: open a run with `mlflow.start_run()`, record **params** (set once), **metrics** (can be logged per `step`), **tags**, and an **artifact** file. Afterwards, `mlflow.search_runs` reads the store back as a DataFrame — the same data the MLflow UI shows.

In [2]:
mlflow.set_experiment("demo-manual")

with mlflow.start_run(run_name="baseline") as run:
    # Params: the knobs you chose (immutable for the run).
    mlflow.log_param("learning_rate", 0.01)
    mlflow.log_param("epochs", 3)

    # Metrics: numbers over time. `step` gives you a curve, not just a final value.
    for epoch in range(3):
        train_loss = round(1.0 / (epoch + 1), 4)   # fake but monotonic
        mlflow.log_metric("train_loss", train_loss, step=epoch)
    mlflow.log_metric("final_accuracy", 0.91)

    # Tags: free-form metadata you can filter on.
    mlflow.set_tag("stage", "experiment")

    # Artifacts: any output file travels with the run.
    report = workdir / "notes.txt"
    report.write_text("Baseline run: small LR, converged cleanly.\n")
    mlflow.log_artifact(str(report))

    run_id = run.info.run_id
    print("Logged run:", run_id)

# Read the run back from the store — exactly what the UI queries.
runs = mlflow.search_runs(experiment_names=["demo-manual"])
print()
print(runs[["run_id", "params.learning_rate", "metrics.final_accuracy", "tags.stage"]].to_string(index=False))

2026/06/23 04:36:39 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/06/23 04:36:39 INFO mlflow.store.db.utils: Updating database tables


2026/06/23 04:36:39 INFO mlflow.tracking.fluent: Experiment with name 'demo-manual' does not exist. Creating a new experiment.


Logged run: 681fb9071ea24a5986bae8d7ef831ad8

                          run_id params.learning_rate  metrics.final_accuracy tags.stage
681fb9071ea24a5986bae8d7ef831ad8                 0.01                    0.91 experiment


### Example 2 — `autolog` + the Model Registry: train, package, version, reload

`mlflow.sklearn.autolog()` captures params, metrics, and the **fitted model** automatically — no manual `log_*` calls. We then **register** the model (turning a run artifact into a named, versioned entry) and reload it through the universal `pyfunc` interface to run inference. This is the full train → package → promote → load path in a dozen lines.

In [3]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

mlflow.set_experiment("demo-sklearn")
mlflow.sklearn.autolog()  # auto-logs params, metrics, AND the fitted model

X, y = load_iris(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)

with mlflow.start_run(run_name="iris-logreg") as run:
    clf = LogisticRegression(max_iter=500).fit(X_tr, y_tr)   # autolog hooks .fit()
    acc = clf.score(X_te, y_te)
    print("Test accuracy:", round(acc, 3))

model_uri = f"runs:/{run.info.run_id}/model"   # where autolog stored the model
print("Model artifact URI:", model_uri)

2026/06/23 04:36:43 INFO mlflow.tracking.fluent: Experiment with name 'demo-sklearn' does not exist. Creating a new experiment.


2026/06/23 04:36:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Matplotlib is building the font cache; this may take a moment.


Test accuracy: 0.974
Model artifact URI: runs:/8d80de4eb386463b8f898040dff54a4d/model


In [4]:
# Promote the logged model into the Model Registry: name + auto-incrementing version.
version = mlflow.register_model(model_uri, name="iris-classifier")
print(f"Registered '{version.name}' as version {version.version}")

# Give that version a deployment-friendly alias instead of hard-coding a number.
client = mlflow.MlflowClient()
client.set_registered_model_alias("iris-classifier", "champion", version.version)

# Load by alias through the universal pyfunc flavor and run inference.
champion = mlflow.pyfunc.load_model("models:/iris-classifier@champion")
preds = champion.predict(X_te[:5])
print("models:/iris-classifier@champion predictions:", preds)
print("Actual labels                              :", y_te[:5])

Successfully registered model 'iris-classifier'.
2026/06/23 04:36:58 WARNING mlflow.tracking._model_registry.fluent: Run with id 8d80de4eb386463b8f898040dff54a4d has no artifacts at artifact path 'model', registering model based on models:/m-fdeeb4f12a244a35b0c548fb0bb37a0a instead


Registered 'iris-classifier' as version 1
models:/iris-classifier@champion predictions: [2 1 0 2 0]
Actual labels                              : [2 1 0 2 0]


Created version '1' of model 'iris-classifier'.


### Optional — logging to a remote tracking server

The same code logs to a shared/remote server when `MLFLOW_TRACKING_URI` points at one (a self-hosted MLflow server, or Databricks). Gated so the notebook still runs end-to-end with no server configured.

In [5]:
remote = os.getenv("MLFLOW_TRACKING_URI", "")
if remote.startswith(("http://", "https://", "databricks")):
    mlflow.set_tracking_uri(remote)
    with mlflow.start_run(run_name="remote-demo"):
        mlflow.log_metric("remote_metric", 1.0)
    print("Logged a run to remote server:", remote)
else:
    print("MLFLOW_TRACKING_URI not set to a server — skipping remote demo.")
    print("Set e.g. MLFLOW_TRACKING_URI=http://localhost:5000 and run `mlflow server` to enable.")

MLFLOW_TRACKING_URI not set to a server — skipping remote demo.
Set e.g. MLFLOW_TRACKING_URI=http://localhost:5000 and run `mlflow server` to enable.


## 6. Gotchas & Pitfalls

- **Params are write-once; metrics are append-many.** Logging the *same* param key twice in a run errors; metrics are designed to be logged repeatedly with a `step`. Don't try to "update" a param — it's a fixed input.
- **The bare `./mlruns` folder store can't do the Model Registry.** `register_model` / aliases require a database-backed store (`sqlite:///…`, Postgres, MySQL) or a server. If registry calls fail locally, that's almost always why.
- **Forgetting to end runs / nested runs.** Use the `with mlflow.start_run():` context manager so the run always closes. A second `start_run()` while one is active creates a *nested* run unless you pass `nested=True` intentionally — orphaned active runs cause confusing "run already active" errors.
- **`autolog()` must be called before `.fit()`**, and you should turn it on for the right flavor (`mlflow.sklearn.autolog()` vs. the broad `mlflow.autolog()`). Calling it after training logs nothing.
- **Missing model signature → schema surprises at serving time.** Log with a `signature`/`input_example` (autolog infers one) so the input/output schema is enforced and documented; otherwise type/column mismatches only surface in production.
- **Environment drift breaks reload.** A logged model pins its dependencies in `requirements.txt`/`conda.yaml`. Loading it in an environment with a very different library version can fail or silently change behavior — recreate the captured env for serving.
- **Tracking URI confusion.** The UI must point at the *same* backend store your code wrote to. Running `mlflow ui` in a different directory (or without `--backend-store-uri`) shows an empty/old store and looks like "my runs vanished."
- **Artifacts vs. backend store are separate.** Metrics/params live in the backend DB; artifacts (the model files) live at the artifact root (local path, S3, etc.). Both must be reachable — a common cause of "run shows up but model won't download."  

## 7. When to Use vs Alternatives

| Tool | Best for | Trade-off vs. MLflow |
|---|---|---|
| **MLflow** | Experiment tracking + model packaging + registry, open-source, self-hostable, framework-agnostic | You run/maintain the server for team use; not a data-versioning or orchestration tool |
| **Weights & Biases** | Polished hosted experiment tracking, rich dashboards, sweeps, collaboration | SaaS-first (self-host is enterprise); see [`weights-and-biases`](weights-and-biases.ipynb) |
| **DVC** | Versioning **data** and pipelines in Git; reproducible DAGs | Not focused on a metrics UI or a model registry — complementary, not a replacement; see [`DVC`](dvc.ipynb) |
| **Optuna** | Hyperparameter **search/optimization** algorithms | Solves *finding* good params; pairs with MLflow for *recording* them; see [`Optuna`](optuna.ipynb) |
| **TensorBoard** | Live training curves & graph/embedding visualization | Visualization only — no params/artifact registry or model packaging |
| **Kubeflow / Sagemaker / Vertex** | End-to-end managed MLOps platforms (pipelines, serving, scaling) | Heavier, cloud/k8s-coupled; MLflow is lighter and embeds into them rather than competing |

**Rule of thumb:** use **MLflow** as the default, vendor-neutral spine for *tracking + model management* — start local with one line, graduate to a shared server when the team needs it. Reach for **W&B** if you want a hosted UI with minimal ops, **DVC/Optuna** alongside MLflow for the jobs it deliberately doesn't do, and a full platform (Kubeflow/SageMaker) when you need orchestration and serving at scale.

## 8. Resources

- **MLflow documentation (home)** — https://mlflow.org/docs/latest/index.html
- **Tracking guide (runs, params, metrics, autolog)** — https://mlflow.org/docs/latest/tracking.html
- **MLflow Models & flavors (packaging, signatures, pyfunc)** — https://mlflow.org/docs/latest/models.html
- **Model Registry (versions, aliases, stages)** — https://mlflow.org/docs/latest/model-registry.html
- **Python API reference** — https://mlflow.org/docs/latest/python_api/index.html
- **GitHub repository** — https://github.com/mlflow/mlflow